In [ ]:
"""
============================================================================
TFM: POBREZA ENERGÉTICA Y FOCALIZACIÓN DEL FOSE EN EL PERÚ
============================================================================
Autor: Carlo Vilches Cevallos | Director: Juan Luis Jiménez (ULPGC)
ERCUE 2019, 2023, 2025 — 11 gráficos de contexto
============================================================================
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'figure.dpi': 150, 'savefig.dpi': 300
})

# 0. CARGAR DATOS
df19 = pd.read_parquet("ERCUE2019.parquet")
df23 = pd.read_parquet("ERCUE2023.parquet")
df25 = pd.read_parquet("base_ercue2025_st.parquet")
print(f"2019: {df19.shape} | 2023: {df23.shape} | 2025: {df25.shape}")

# 1. FUNCIONES AUXILIARES (ponderadas por factor de expansión)
def weighted_median(values, weights):
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    v, w = values[mask], weights[mask]
    if len(v) == 0: return np.nan
    idx = np.argsort(v); v, w = v[idx], w[idx]
    cum = np.cumsum(w) / np.sum(w)
    return v[np.searchsorted(cum, 0.5)]

def weighted_mean(values, weights):
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    v, w = values[mask], weights[mask]
    if len(v) == 0: return np.nan
    return np.average(v, weights=w)

def weighted_pct(cond, weights):
    mask = np.isfinite(weights.values) & (weights.values > 0) & np.isfinite(cond.astype(float).values)
    c, w = cond.astype(float).values[mask], weights.values[mask]
    if len(c) == 0: return np.nan
    return np.average(c, weights=w) * 100

# 2. PREPARACIÓN DE VARIABLES
# --- 2019 (pre-reforma, umbral 100 kWh) ---
df19['consumo'] = df19['conskwprom'].astype(float)
df19['gasto_elec'] = pd.to_numeric(df19['Y23_5F'], errors='coerce')
df19['gasto_tot_mensual'] = df19['gasto_total'].astype(float) / 12
df19['eb'] = df19['gasto_elec'] / df19['gasto_tot_mensual']
df19['peso'] = df19['Fact_exp_hog'].astype(float)
df19['area_std'] = df19['AREA_MUE'].map({'Urbana': 'Urbana', 'Rural': 'Rural'})
df19['pobreza_std'] = df19['pobreza']
df19['year'] = 2019

# --- 2023 (post-reforma, umbral 140 kWh) ---
df23['consumo'] = df23['conskwprom'].astype(float)
df23['gasto_elec'] = pd.to_numeric(df23['p4_20_6'], errors='coerce')
df23['gasto_tot_mensual'] = df23['gasto_total'].astype(float) / 12
df23['eb'] = df23['gasto_elec'] / df23['gasto_tot_mensual']
df23['peso'] = df23['Fact_exp_hog'].astype(float)
df23['area_std'] = df23['AREA1'].map({'Urbana': 'Urbana', 'Rural': 'Rural'})
df23['pobreza_std'] = df23['POBREZA']
df23['year'] = 2023

# --- 2025 (post-reforma, segundo relevamiento) ---
df25['consumo'] = df25['consumo_fose'].astype(float)
df25['gasto_elec'] = df25['gasto_mes_electricidad'].astype(float)
df25['gasto_tot_mensual'] = df25['ga_aj_total_pc'].astype(float) * df25['N_PERSONAS'].astype(float)
df25['eb'] = df25['gasto_elec'] / df25['gasto_tot_mensual']
df25['peso'] = df25['Fact_exp_hog'].astype(float)
df25['area_std'] = df25['area'].map({'Urbana': 'Urbana', 'Rural': 'Rural'})
df25['pobreza_std'] = df25['POBREZA']
df25['year'] = 2025

# Limpiar EB inválidos
for df in [df19, df23, df25]:
    df.loc[(df['eb'] <= 0) | (df['eb'] > 1) | ~np.isfinite(df['eb']), 'eb'] = np.nan

# MULTIPREDIO (solo 2025): suministros duplicados = compartidos
mask_valido = (df25['p5_6_2_1'].astype(str).str.strip().str.len() > 2) & \
              (df25['p5_6_2_1'].astype(str) != 'NO SE VE')
df25['multipredio'] = False
duplicados = df25.loc[mask_valido, 'p5_6_2_1'].duplicated(keep=False)
df25.loc[mask_valido, 'multipredio'] = duplicados.values
print(f"\nMultipredio: {df25['multipredio'].sum()} ({df25['multipredio'].sum()/mask_valido.sum()*100:.1f}%)")

# SECTOR TÍPICO (solo 2025)
st_var = None
for c in ['sector_tipico', 'SECTOR_TIPICO', 'st', 'ST', 'sector_t', 'SECTOR']:
    if c in df25.columns: st_var = c; break
if st_var is None:
    cands = [c for c in df25.columns if 'sector' in c.lower() or 'tipico' in c.lower()]
    if cands: st_var = cands[0]
if st_var: print(f"Sector típico: '{st_var}'"); print(df25[st_var].value_counts())

dfs_list = [df19, df23, df25]
years = [2019, 2023, 2025]
xlabs = ['2019\n(pre-reforma)', '2023\n(post-reforma)', '2025\n(post-reforma)']
print("\n✓ Variables preparadas\n" + "="*60)

# ############################################################
# 1/11: EVOLUCIÓN CONSUMO + ENERGY BURDEN
# ############################################################
stats_e = {}
for lab, ff in [('Nacional', lambda d: d), ('Urbano', lambda d: d[d['area_std']=='Urbana']),
                ('Rural', lambda d: d[d['area_std']=='Rural'])]:
    stats_e[f'cons_{lab}'] = [weighted_median(ff(df).dropna(subset=['consumo','peso'])['consumo'].values,
        ff(df).dropna(subset=['consumo','peso'])['peso'].values) for df in dfs_list]
for lab, ff in [('Nacional', lambda d: d), ('Pobre extremo', lambda d: d[d['pobreza_std']=='Pobre extremo']),
    ('Pobre no extremo', lambda d: d[d['pobreza_std']=='Pobre no extremo']),
    ('No pobre', lambda d: d[d['pobreza_std']=='No pobre'])]:
    stats_e[f'eb_{lab}'] = [weighted_median(ff(df).dropna(subset=['eb','peso'])['eb'].values,
        ff(df).dropna(subset=['eb','peso'])['peso'].values)*100 for df in dfs_list]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
ca = {'Nacional':'#2c3e50','Urbano':'#2980b9','Rural':'#27ae60'}
ma_d = {'Nacional':'o','Urbano':'s','Rural':'D'}
for l in ['Nacional','Urbano','Rural']:
    v = stats_e[f'cons_{l}']
    ax1.plot(years, v, marker=ma_d[l], color=ca[l], lw=2, ms=8, label=l)
    for x, y in zip(years, v):
        ax1.annotate(f'{y:.0f}', (x,y), textcoords="offset points",
                     xytext=(8, 8 if l!='Rural' else -15), fontsize=9, color=ca[l], fontweight='bold')
ax1.axhline(100, color='red', ls='--', alpha=0.4); ax1.axhline(140, color='red', ls='-.', alpha=0.4)
ax1.text(2019.1, 101, '100 kWh\n(hasta 2022)', fontsize=7, color='red', alpha=0.6)
ax1.text(2023.1, 141, '140 kWh\n(desde 2022)', fontsize=7, color='red', alpha=0.6)
ax1.axvline(2022, color='gray', ls=':', alpha=0.5); ax1.text(2022.1, 150, 'Reforma\nFOSE', fontsize=7, color='gray')
ax1.set_title('Panel A. Consumo eléctrico mediano por área'); ax1.set_ylabel('kWh/mes')
ax1.set_xticks(years); ax1.legend(fontsize=9); ax1.set_ylim(0,160); ax1.grid(axis='y', alpha=0.3)

cb = {'Pobre extremo':'#c0392b','Pobre no extremo':'#e67e22','Nacional':'#2c3e50','No pobre':'#2980b9'}
mb = {'Pobre extremo':'^','Pobre no extremo':'v','Nacional':'o','No pobre':'s'}
for l in ['Pobre extremo','Pobre no extremo','Nacional','No pobre']:
    v = stats_e[f'eb_{l}']
    ax2.plot(years, v, marker=mb[l], color=cb[l], lw=2, ms=8, label=l)
    for x, y in zip(years, v):
        ax2.annotate(f'{y:.2f}%', (x,y), textcoords="offset points",
                     xytext=(8, 8 if l!='No pobre' else -15), fontsize=8, color=cb[l], fontweight='bold')
ax2.axvline(2022, color='gray', ls=':', alpha=0.5)
ax2.set_title('Panel B. Energy burden mediano por condición de pobreza'); ax2.set_ylabel('Energy burden mediano (%)')
ax2.set_xticks(years); ax2.legend(loc='upper left', fontsize=9); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('evolucion_consumo_eb.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 1/11")

# ############################################################
# 2/11: GRÁFICO 11 — Distribución del consumo eléctrico mensual y umbral FOSE
# ############################################################
fig, (ax_u, ax_r) = plt.subplots(2, 1, figsize=(12, 9))
mask_c = df25['consumo'].notna() & (df25['consumo']>0) & (df25['consumo']<350) & df25['peso'].notna()
dc = df25[mask_c].copy()
x_grid = np.linspace(0, 250, 500)
for ax, area, cfill, cline in [(ax_u,'Urbana','#3498db','#2980b9'),(ax_r,'Rural','#e74c3c','#c0392b')]:
    sub = dc[dc['area_std']==area]
    kde = gaussian_kde(sub['consumo'].values, bw_method=0.15, weights=sub['peso'].values)
    d = kde(x_grid)
    ax.fill_between(x_grid, d, alpha=0.3, color=cfill); ax.plot(x_grid, d, color=cline, lw=2, label=area)
    kde_n = gaussian_kde(dc['consumo'].values, bw_method=0.15, weights=dc['peso'].values)
    ax.plot(x_grid, kde_n(x_grid), color='#2c3e50', lw=1.5, ls='--', label='Nacional')
    for a,b,col in [(0,30,'#27ae60'),(30,100,'#3498db'),(100,140,'#9b59b6'),(140,250,'#e74c3c')]:
        ax.axvspan(a, b, alpha=0.06, color=col)
    ax.axvline(30, color='#27ae60', ls='--', alpha=0.6); ax.axvline(100, color='#9b59b6', ls='-.', alpha=0.6)
    ax.axvline(140, color='#e74c3c', ls='-', alpha=0.8, lw=1.5)
    med = weighted_median(sub['consumo'].values, sub['peso'].values)
    ax.axvline(med, color=cline, ls=':', alpha=0.7, lw=1.5)
    ax.annotate(f'Mediana\n{area.lower()}\n{med:.0f} kWh', xy=(med, d.max()*0.5), fontsize=8, color=cline, fontstyle='italic',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=cline, alpha=0.7))
    for xv, lb, co in [(30,'30 kWh','#27ae60'),(100,'100 kWh','#9b59b6'),(140,'140 kWh','#e74c3c')]:
        ax.text(xv, d.max()*0.95, lb, fontsize=8, color=co, ha='center', va='top',
                bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec=co, alpha=0.8))
    ax.set_ylabel('Densidad'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.2)
mn = weighted_median(dc['consumo'].values, dc['peso'].values)
p30 = weighted_pct(dc['consumo']<=30, dc['peso']); p140 = weighted_pct(dc['consumo']<=140, dc['peso'])
p101 = weighted_pct((dc['consumo']>100)&(dc['consumo']<=140), dc['peso'])
ax_u.text(0.98, 0.55, f'Mediana nacional: {mn:.0f} kWh\n% ≤ 30 kWh (B1): {p30:.1f}%\n'
    f'% ≤ 140 kWh (benef.): {p140:.1f}%\n% 101–140 kWh (nuevo): {p101:.1f}%',
    transform=ax_u.transAxes, fontsize=8, va='top', ha='right', bbox=dict(boxstyle='round', fc='wheat', alpha=0.8))
ax_r.set_xlabel('Consumo eléctrico mensual (kWh)')
plt.suptitle('Gráfico 11. Distribución del consumo eléctrico mensual con umbrales FOSE\nERCUE 2025',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('grafico1_distribucion_consumo.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 2/11")

# ############################################################
# 3/11: CUADRO CONSUMO POR SECTOR TÍPICO (solo 2025)
# ############################################################
if st_var:
    ms = df25['consumo'].notna() & df25[st_var].notna() & df25['peso'].notna(); ds = df25[ms].copy()
    rows = []
    for st in ['ST1','ST2','ST3','ST4','SER']:
        s = ds[ds[st_var].astype(str).str.upper().str.strip()==st]
        if len(s)==0: s = ds[ds[st_var].astype(str).str.contains(st, case=False, na=False)]
        if len(s)>0:
            rows.append([st, f'{weighted_mean(s["consumo"].values,s["peso"].values):.1f}',
                f'{weighted_median(s["consumo"].values,s["peso"].values):.1f}',
                f'{weighted_pct(s["consumo"]<=30,s["peso"]):.1f}',
                f'{weighted_pct((s["consumo"]>30)&(s["consumo"]<=140),s["peso"]):.1f}',
                f'{weighted_pct(s["consumo"]>140,s["peso"]):.1f}',
                f'{weighted_pct(s["pobreza_std"].isin(["Pobre extremo","Pobre no extremo"]),s["peso"]):.1f}'])
    c1 = pd.DataFrame(rows, columns=['Sector Típico','Consumo prom\n(kWh/mes)','Mediana\n(kWh/mes)',
        '% ≤30 kWh','% 31–140 kWh','% >140 kWh','Pobreza (%)'])
    fig, ax = plt.subplots(figsize=(12,3)); ax.axis('off')
    t = ax.table(cellText=c1.values, colLabels=c1.columns, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1,1.8)
    for j in range(len(c1.columns)): t[0,j].set_facecolor('#2c3e50'); t[0,j].set_text_props(color='white', fontweight='bold')
    for i in range(1,len(c1)+1):
        for j in range(len(c1.columns)):
            if i%2==0: t[i,j].set_facecolor('#ecf0f1')
    plt.title('Cuadro 1. Consumo por sector típico — ERCUE 2025', fontsize=12, fontweight='bold', pad=20)
    plt.savefig('cuadro1_consumo_sector.png', dpi=300, bbox_inches='tight'); plt.show(); print("✓ 3/11")
else: print("⚠ 3/11 OMITIDO")

# ############################################################
# 4/11: GRÁFICO 4 — Precio efectivo implícito por kWh
# ############################################################
if st_var:
    dp = df25[ms].copy(); dp['precio'] = dp['gasto_elec'] / dp['consumo']
    dp = dp[(dp['precio']>0) & (dp['precio']<dp['precio'].quantile(0.99))]
    secs = ['ST1','ST2','ST3','ST4','SER']
    cst = {'ST1':'#3498db','ST2':'#2ecc71','ST3':'#f39c12','ST4':'#e74c3c','SER':'#9b59b6'}
    lst = {'ST1':'ST1 (Lima)','ST2':'ST2 (Urb. medio)','ST3':'ST3 (Urb-rural)','ST4':'ST4 (Rural)','SER':'SER (Sist. rurales)'}
    fig, ax = plt.subplots(figsize=(10,6)); db, meds = [], []
    for st in secs:
        s = dp[dp[st_var].astype(str).str.upper().str.strip()==st]
        if len(s)==0: s = dp[dp[st_var].astype(str).str.contains(st, case=False, na=False)]
        db.append(s['precio'].values if len(s)>0 else [0]); meds.append(np.median(s['precio'].values) if len(s)>0 else 0)
    bp = ax.boxplot(db, positions=range(1,len(secs)+1), widths=0.6, patch_artist=True,
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for p, st in zip(bp['boxes'], secs): p.set_facecolor(cst[st]); p.set_alpha(0.6)
    mn_p = np.median(dp['precio'].values)
    ax.axhline(mn_p, color='#2c3e50', ls='--', alpha=0.6)
    ax.text(len(secs)+0.3, mn_p, f'Mediana nac.\nS/ {mn_p:.3f}/kWh', fontsize=8, color='#2c3e50', va='center')
    for i, (pos, med) in enumerate(zip(range(1,len(secs)+1), meds)):
        ax.annotate(f'S/ {med:.3f}', (pos,med), textcoords="offset points", xytext=(0,-25), fontsize=9,
                    fontweight='bold', ha='center', bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8))
    ax.set_xticklabels(secs); ax.set_xlabel('Sector Típico'); ax.set_ylabel('Precio implícito (S//kWh)')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=cst[s], alpha=0.6, label=lst[s]) for s in secs], loc='upper left', fontsize=8)
    ax.set_title('Gráfico 4. Precio efectivo implícito por kWh\nERCUE 2025', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.savefig('grafico4_precio_implicito.png', dpi=300, bbox_inches='tight'); plt.show(); print("✓ 4/11")
else: print("⚠ 4/11 OMITIDO")

# ############################################################
# 5/11: CUADRO INDICADORES PE
# ############################################################
me = df25['eb'].notna() & df25['peso'].notna() & (df25['eb']>0) & (df25['eb']<1)
de = df25[me].copy()
med_eb = weighted_median(de['eb'].values, de['peso'].values); umb_2m = med_eb * 2
de['sob'] = (de['eb'] > umb_2m).astype(int)
for av in ['Urbana','Rural']:
    sa = de[de['area_std']==av]; mg = weighted_median(sa['gasto_elec'].values, sa['peso'].values)
    de.loc[de['area_std']==av, 'umb_hep'] = mg * 0.5
de['hep'] = (de['gasto_elec'] < de['umb_hep']).astype(int)
print(f"\nMediana EB: {med_eb*100:.2f}% → Umbral 2M: {umb_2m*100:.2f}%")

rpe = []
for lab, s in [('Nacional',de),('Urbano',de[de['area_std']=='Urbana']),('Rural',de[de['area_std']=='Rural']),
    ('Pobre extremo',de[de['pobreza_std']=='Pobre extremo']),('Pobre no extremo',de[de['pobreza_std']=='Pobre no extremo']),
    ('No pobre',de[de['pobreza_std']=='No pobre'])]:
    rpe.append([lab, f'{weighted_median(s["eb"].values,s["peso"].values)*100:.2f}', f'{umb_2m*100:.2f}',
        f'{weighted_pct(s["sob"]==1,s["peso"]):.1f}', f'{weighted_pct(s["hep"]==1,s["peso"]):.1f}'])
c2 = pd.DataFrame(rpe, columns=['Grupo','Mediana EB (%)','Umbral 2M (%)','Tasa 2M (%)','Tasa HEP (%)'])
fig, ax = plt.subplots(figsize=(10,3.5)); ax.axis('off')
t = ax.table(cellText=c2.values, colLabels=c2.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1,1.8)
for j in range(len(c2.columns)): t[0,j].set_facecolor('#2c3e50'); t[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,len(c2)+1):
    for j in range(len(c2.columns)):
        if i%2==0: t[i,j].set_facecolor('#ecf0f1')
plt.title('Cuadro 2. Indicadores de pobreza energética — ERCUE 2025', fontsize=12, fontweight='bold', pad=20)
plt.savefig('cuadro2_indicadores_pe.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 5/11")

# ############################################################
# 6/11: GRÁFICO 2 — Indicadores de pobreza energética por grupo
# ############################################################
gl = ['Nacional','Urbano','Rural','Pobre\nextremo','Pobre no\nextremo','No pobre']
fp = [de, de[de['area_std']=='Urbana'], de[de['area_std']=='Rural'],
    de[de['pobreza_std']=='Pobre extremo'], de[de['pobreza_std']=='Pobre no extremo'],
    de[de['pobreza_std']=='No pobre']]
t2 = [weighted_pct(s['sob']==1,s['peso']) for s in fp]
th = [weighted_pct(s['hep']==1,s['peso']) for s in fp]
fig, ax = plt.subplots(figsize=(12,6)); x = np.arange(len(gl)); w = 0.35
b1 = ax.bar(x-w/2, t2, w, label='Tasa 2M (%)', color='#2c6fbb', zorder=3)
b2 = ax.bar(x+w/2, th, w, label='Tasa HEP (%)', color='#d4756b', zorder=3)
for bar, val in zip(b1, t2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#2c6fbb', fontweight='bold')
for bar, val in zip(b2, th):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#d4756b', fontweight='bold')
ax.axhline(t2[0], color='#2c6fbb', ls='--', alpha=0.3); ax.axvline(2.5, color='gray', ls=':', alpha=0.4)
ax.text(1, 48, 'Por área', fontsize=9, ha='center', fontstyle='italic', color='gray')
ax.text(4, 48, 'Por condición de pobreza', fontsize=9, ha='center', fontstyle='italic', color='gray')
ax.set_xticks(x); ax.set_xticklabels(gl); ax.set_ylabel('Porcentaje (%)'); ax.set_ylim(0,50)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_title('Gráfico 2. Indicadores de pobreza energética por grupo — ERCUE 2025', fontsize=13, fontweight='bold')
fig.text(0.5, -0.06, f'Umbral 2M = {umb_2m*100:.2f}% | HEP: gasto elec < 50% mediana por área', ha='center', fontsize=8, color='gray')
plt.tight_layout(); plt.savefig('grafico2_pe_por_grupo.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 6/11")

# ############################################################
# 7/11: EVOLUCIÓN PE 2019-2025
# ############################################################
pev = {}
for df in dfs_list:
    m = df['eb'].notna() & df['peso'].notna() & (df['eb']>0) & (df['eb']<1); d = df[m].copy()
    mn = weighted_median(d['eb'].values, d['peso'].values); u = mn*2; d['s2m'] = (d['eb']>u).astype(int)
    for av in ['Urbana','Rural']:
        sa = d[d['area_std']==av]
        if len(sa)>0: d.loc[d['area_std']==av, 'uh'] = weighted_median(sa['gasto_elec'].values, sa['peso'].values)*0.5
    d['hp'] = (d['gasto_elec']<d['uh']).astype(int)
    for lab, ff in [('Nacional',lambda x:x),('Urbano',lambda x:x[x['area_std']=='Urbana']),
        ('Rural',lambda x:x[x['area_std']=='Rural']),('Pobre extremo',lambda x:x[x['pobreza_std']=='Pobre extremo']),
        ('Pobre no extremo',lambda x:x[x['pobreza_std']=='Pobre no extremo']),('No pobre',lambda x:x[x['pobreza_std']=='No pobre'])]:
        s = ff(d)
        if len(s)>0:
            pev.setdefault(f'2m_{lab}',[]).append(weighted_pct(s['s2m']==1,s['peso']))
            pev.setdefault(f'hep_{lab}',[]).append(weighted_pct(s['hp']==1,s['peso']))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16,6))
cg = {'Pobre extremo':'#c0392b','Pobre no extremo':'#e67e22','Urbano':'#2980b9','Nacional':'#2c3e50','No pobre':'#27ae60','Rural':'#7f8c8d'}
mg2 = {'Pobre extremo':'^','Pobre no extremo':'v','Urbano':'s','Nacional':'o','No pobre':'D','Rural':'D'}
lg2 = {'Pobre extremo':'-','Pobre no extremo':'-','Urbano':'--','Nacional':'--','No pobre':':','Rural':':'}
for ind, ax, ti in [('2m',a1,'Panel A. Tasa 2M\n(EB > 2 × mediana nacional)'),
                     ('hep',a2,'Panel B. Tasa HEP\n(gasto eléc < 50% mediana por área)')]:
    for lab in ['Pobre extremo','Pobre no extremo','Urbano','Nacional','No pobre','Rural']:
        k = f'{ind}_{lab}'
        if k in pev and len(pev[k])==3:
            v = pev[k]; ax.plot(years, v, marker=mg2[lab], color=cg[lab], lw=2, ms=7, label=lab, ls=lg2[lab])
            ax.annotate(f'{v[0]:.1f}%', (2019,v[0]), textcoords="offset points", xytext=(-30,5), fontsize=7, color=cg[lab])
            ax.annotate(f'{lab} {v[-1]:.1f}%', (2025,v[-1]), textcoords="offset points", xytext=(8,0), fontsize=7, color=cg[lab])
    ax.axvline(2022, color='gray', ls=':', alpha=0.5); ax.text(2022.05, 45, 'Reforma\nFOSE', fontsize=7, color='gray')
    ax.set_title(ti, fontsize=11, fontweight='bold'); ax.set_xticks(years); ax.set_xticklabels(xlabs, fontsize=9)
    ax.set_ylabel('%'); ax.set_ylim(0,50); ax.grid(axis='y', alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.suptitle('Evolución indicadores PE, 2019–2025', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('evolucion_pe.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 7/11")

# ############################################################
# 8/11: EVOLUCIÓN COBERTURA + ERRORES
# ############################################################
def fose_s(df, umb):
    m = df['consumo'].notna() & df['peso'].notna() & df['pobreza_std'].notna(); d = df[m].copy()
    d['el'] = (d['consumo']<=umb).astype(int); d['po'] = d['pobreza_std'].isin(['Pobre extremo','Pobre no extremo']).astype(int)
    cn = weighted_pct(d['el']==1, d['peso'])
    cu = weighted_pct(d.loc[d['area_std']=='Urbana','el']==1, d.loc[d['area_std']=='Urbana','peso'])
    cr = weighted_pct(d.loc[d['area_std']=='Rural','el']==1, d.loc[d['area_std']=='Rural','peso'])
    el = d[d['el']==1]; ei = weighted_pct(el['po']==0, el['peso'])
    po = d[d['po']==1]; ee = weighted_pct(po['el']==0, po['peso'])
    return {'cn':cn,'cu':cu,'cr':cr,'ei':ei,'ee':ee}

s19 = fose_s(df19, 100); s23 = fose_s(df23, 140); s25 = fose_s(df25, 140)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14,5.5))
for lab, k, col, mk in [('Nacional','cn','#2c3e50','o'),('Urbano','cu','#2980b9','s'),('Rural','cr','#27ae60','D')]:
    v = [s19[k], s23[k], s25[k]]; a1.plot(years, v, marker=mk, color=col, lw=2, ms=8, label=lab)
    for x, y in zip(years, v):
        a1.annotate(f'{y:.1f}%', (x,y), textcoords="offset points", xytext=(8,5), fontsize=9, color=col, fontweight='bold')
a1.axvline(2022, color='gray', ls=':', alpha=0.5); a1.text(2022.1, 55, 'Reforma\nFOSE', fontsize=8, color='gray')
a1.annotate('Umbral:\n100 kWh', (2019.3,48), fontsize=7, color='purple', fontstyle='italic')
a1.annotate('Umbral:\n140 kWh', (2023.3,55), fontsize=7, color='purple', fontstyle='italic')
a1.set_title('Panel A. Cobertura FOSE por área'); a1.set_ylabel('% hogares bajo umbral')
a1.set_xticks(years); a1.set_xticklabels(xlabs, fontsize=9); a1.set_ylim(40,102); a1.legend(loc='lower right', fontsize=9); a1.grid(axis='y', alpha=0.3)

eiv = [s19['ei'],s23['ei'],s25['ei']]; eev = [s19['ee'],s23['ee'],s25['ee']]
a2.plot(years, eiv, marker='o', color='#8b0000', lw=2, ms=8, label='Error inclusión\n(benef no pobres/total benef)')
a2.plot(years, eev, marker='s', color='#2c6fbb', lw=2, ms=8, label='Error exclusión\n(pobres excluidos/total pobres)')
for x, y in zip(years, eiv): a2.annotate(f'{y:.1f}%', (x,y), textcoords="offset points", xytext=(8,5), fontsize=9, color='#8b0000', fontweight='bold')
for x, y in zip(years, eev): a2.annotate(f'{y:.1f}%', (x,y), textcoords="offset points", xytext=(8,-15), fontsize=9, color='#2c6fbb', fontweight='bold')
a2.axvline(2022, color='gray', ls=':', alpha=0.5); a2.text(2022.1, 75, 'Reforma\nFOSE', fontsize=8, color='gray')
a2.set_title('Panel B. Errores de focalización'); a2.set_ylabel('Porcentaje (%)')
a2.set_xticks(years); a2.set_xticklabels(xlabs, fontsize=9); a2.legend(loc='center right', fontsize=8); a2.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('evolucion_cobertura_errores.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 8/11")

# ############################################################
# 9/11: CUADRO COBERTURA POR POBREZA
# ############################################################
mf = df25['consumo'].notna() & df25['peso'].notna() & df25['pobreza_std'].notna()
dff = df25[mf].copy(); dff['el'] = (dff['consumo']<=140).astype(int)
rcb = []
for lab, f in [('Pobre extremo',dff[dff['pobreza_std']=='Pobre extremo']),
    ('Pobre no extremo',dff[dff['pobreza_std']=='Pobre no extremo']),
    ('No pobre',dff[dff['pobreza_std']=='No pobre']),('Total',dff)]:
    r = weighted_pct(f['el']==1, f['peso']); rcb.append([lab, f'{r:.1f}', f'{100-r:.1f}'])
ccb = pd.DataFrame(rcb, columns=['Condición de pobreza','Recibe FOSE (%)','No recibe FOSE (%)'])
fig, ax = plt.subplots(figsize=(8,2.5)); ax.axis('off')
t = ax.table(cellText=ccb.values, colLabels=ccb.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1,1.8)
for j in range(len(ccb.columns)): t[0,j].set_facecolor('#2c3e50'); t[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,len(ccb)+1):
    for j in range(len(ccb.columns)):
        if i%2==0: t[i,j].set_facecolor('#ecf0f1')
plt.title('Cobertura del FOSE por condición de pobreza — ERCUE 2025', fontsize=12, fontweight='bold', pad=20)
plt.savefig('cuadro_cobertura_pobreza.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 9/11")

# ############################################################
# 10/11: CUADRO DESCOMPOSICIÓN ERROR DE EXCLUSIÓN
# ############################################################
# Exclusión secuencial: 1) consumo>140, 2) multipredio, 3) estrato Lima
dex = df25[mf].copy()
dex['pob'] = dex['pobreza_std'].isin(['Pobre extremo','Pobre no extremo']).astype(int)
dex['ec'] = (dex['consumo']>140).astype(int); dex['em'] = dex['multipredio'].astype(int)
pob = dex[dex['pob']==1].copy(); tp = pob['peso'].sum()
pc = pob[pob['ec']==1]['peso'].sum() / tp * 100
pm = pob[(pob['ec']==0) & (pob['em']==1)]['peso'].sum() / tp * 100
print(f"\nDescomposición EE: consumo={pc:.1f}%, multi={pm:.1f}%, total={pc+pm:.1f}%")

ree = []
for clab, cc in [('Consumo > 140 kWh','c'),('Multipredio','m')]:
    row = [clab]
    for af in [pob, pob[pob['area_std']=='Urbana'], pob[pob['area_std']=='Rural']]:
        tt = af['peso'].sum()
        if cc=='c': val = af[af['ec']==1]['peso'].sum()/tt*100
        else: val = af[(af['ec']==0)&(af['em']==1)]['peso'].sum()/tt*100
        row.append(f'{val:.1f}')
    ree.append(row)
ree.append(['Estrato manzana (Lima)','0','0','0'])
cee = pd.DataFrame(ree, columns=['Criterio','% excl. total','% excl. urbano','% excl. rural'])
fig, ax = plt.subplots(figsize=(10,2.5)); ax.axis('off')
t = ax.table(cellText=cee.values, colLabels=cee.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1,1.8)
for j in range(len(cee.columns)): t[0,j].set_facecolor('#2c3e50'); t[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,len(cee)+1):
    for j in range(len(cee.columns)):
        if i%2==0: t[i,j].set_facecolor('#ecf0f1')
plt.title('Descomposición del error de exclusión — ERCUE 2025', fontsize=12, fontweight='bold', pad=20)
plt.savefig('cuadro_descomposicion_ee.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 10/11")

# ############################################################
# 11/11: STACKED BARS DESCOMPOSICIÓN EE
# ############################################################
fig, ax = plt.subplots(figsize=(8,6))
for idx, cat in enumerate(['Pobre extremo','Pobre no extremo']):
    s = pob[pob['pobreza_std']==cat]; tt = s['peso'].sum()
    vc = s[s['ec']==1]['peso'].sum()/tt*100; vm = s[(s['ec']==0)&(s['em']==1)]['peso'].sum()/tt*100
    ax.bar(idx, vc, color='#e74c3c', alpha=0.8, label='Consumo >140 kWh' if idx==0 else '')
    ax.bar(idx, vm, bottom=vc, color='#f39c12', alpha=0.8, label='Multipredio' if idx==0 else '')
    ax.bar(idx, 0, bottom=vc+vm, color='#9b59b6', alpha=0.8, label='Estrato Lima' if idx==0 else '')
    if vc>1: ax.text(idx, vc/2, f'{vc:.1f}%', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    if vm>1: ax.text(idx, vc+vm/2, f'{vm:.1f}%', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.set_xticks(range(2)); ax.set_xticklabels(['Pobre extremo','Pobre no extremo'], fontsize=11)
ax.set_ylabel('% hogares pobres excluidos'); ax.set_ylim(0,20); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_title('Descomposición error de exclusión\nERCUE 2025', fontsize=13, fontweight='bold')
fig.text(0.5, -0.04, f'Error de exclusión total: {pc+pm:.1f}%', ha='center', fontsize=8, color='gray')
plt.tight_layout(); plt.savefig('grafico_descomposicion_ee.png', dpi=300, bbox_inches='tight'); plt.show()
print("✓ 11/11")

print("\n" + "="*60)
print("✅ TODOS LOS GRÁFICOS GENERADOS")
print("="*60)

In [ ]:
import os
from google.colab import files

for f in os.listdir('/content/'):
    if f.endswith('.png'):
        files.download(f'/content/{f}')
        print(f"Descargado: {f}")